# 10주차 이론 · 데이터분석가처럼 문제 정의하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phyzikyh/computing/blob/main/%EA%B0%95%EC%9D%98%EA%B5%90%EC%9E%AC/week10_lecture.ipynb)

카페 주문 기록은 수업용으로 만든 가상 자료입니다.

## 1. 분석 질문

### 1.1 모호한 문제를 분석 질문으로

모호한 문제 "요즘 카페가 너무 느립니다"를 다섯 요소(대상, 측정값, 기준, 계산, 범위)로 구체화하면 다음 질문이 됩니다.

> 교내 카페의 주문 시간대별 평균 대기시간은 어떻게 다르며, 어느 시간대에 대기시간이 가장 긴가?

- 대상 주문 한 건 / 측정값 `wait_minutes` / 기준 `order_hour` / 계산 평균 / 범위 이 자료의 5일간
- 아래 코드는 이 질문에 답하는 데 필요한 순서(구조 확인, 품질 확인, 요약, 그룹별 비교)로 실행합니다.

## 2. 데이터 불러오기

### 2.1 CSV 파일 읽기

CSV 파일을 DataFrame으로 읽어 `data`에 저장합니다. 아래 셀은 이론의 `pd.read_csv("data/week10_cafe_orders.csv")`와 같은 일을 하되, 이 노트북의 위치나 Colab에서도 파일을 찾도록 경로를 확인합니다.

In [1]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("data/week10_cafe_orders.csv"),
    Path("../강의교재/data/week10_cafe_orders.csv"),
    Path("강의교재/data/week10_cafe_orders.csv"),
]
url = "https://raw.githubusercontent.com/phyzikyh/computing/main/%EA%B0%95%EC%9D%98%EA%B5%90%EC%9E%AC/data/week10_cafe_orders.csv"

data_path = next((p for p in candidates if p.exists()), None)
data = pd.read_csv(data_path if data_path else url)
print("불러온 파일:", data_path if data_path else url)
data.shape

불러온 파일: ..\강의교재\data\week10_cafe_orders.csv


(39, 8)

**코드 설명**
- `import pandas as pd`는 pandas를 불러오며, `pd`는 관례적인 짧은 이름입니다.
- `candidates`는 파일이 있을 만한 경로의 목록이고, `p.exists()`로 실제 있는지 확인해 처음 발견한 경로를 `data_path`에 둡니다. 없으면 `None`이 되어 `url`을 씁니다.
- `pd.read_csv()`는 쉼표로 구분된 CSV 파일을 읽어 DataFrame으로 만들고, 그 결과를 `data`에 저장합니다.
- 마지막 줄 `data.shape`는 `(행 수, 열 수)`를 보여 줍니다.

**결과 해석**
- `data`에는 파일 이름이 아니라 CSV에서 읽은 DataFrame이 저장됩니다.
- `(39, 8)`은 39행 8열, 즉 주문 39건에 변수 8개라는 뜻입니다.

## 3. Series

DataFrame은 1차원 Series를 여러 개 묶은 표입니다. 월요일 주문 8건의 대기시간을 주문 번호를 레이블로 붙여 Series로 만들어 봅니다.

### 3.1 Series 만들기

In [2]:
wait = pd.Series(
    [3.2, 4.1, 5.4, 9.2, 10.5, 7.3, 4.0, 4.8],
    index=[1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
)
wait

1001     3.2
1002     4.1
1003     5.4
1004     9.2
1005    10.5
1006     7.3
1007     4.0
1008     4.8
dtype: float64

**코드 설명**
- `pd.Series(값의 목록, index=레이블의 목록)`으로 Series를 만듭니다. 값 8개와 레이블 8개가 순서대로 짝을 이룹니다.
- 만든 Series를 `wait`에 저장하고, 마지막 줄 `wait`로 내용을 출력합니다.

**결과 해석**
- 왼쪽 주문 번호가 레이블 인덱스, 오른쪽이 값입니다.
- 값이 모두 소수라서 자료형은 `float64`입니다.

### 3.2 Series 정보 확인

In [3]:
print("레이블:", wait.index.tolist())
print("값의 개수:", wait.size)
print("형태:", wait.shape)
print("자료형:", wait.dtype)

레이블: [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008]
값의 개수: 8
형태: (8,)
자료형: float64


**코드 설명**
- `wait.index`는 레이블 전체이고, `.tolist()`는 이를 파이썬 리스트로 바꿉니다.
- `wait.size`는 값의 개수, `wait.shape`는 배열의 형태, `wait.dtype`은 자료형입니다. 이들은 속성이므로 괄호 없이 씁니다.

**결과 해석**
- 값이 8개인 1차원 `float64`이고, 레이블은 1001~1008입니다.

### 3.3 레이블과 절대 위치

`iloc`은 절대 위치(0부터), `loc`는 레이블로 값을 가리킵니다. 실행 전에 두 값을 예상해 봅니다.

In [4]:
print(wait.iloc[0])
print(wait.loc[1003])

3.2
5.4


**코드 설명**
- `wait.iloc[0]`은 맨 앞(위치 0)의 값을 가져옵니다.
- `wait.loc[1003]`은 레이블이 1003인 값을 가져옵니다.

**결과 해석**
- `iloc[0]`은 3.2, `loc[1003]`은 1003번 주문의 값 5.4입니다.
- 레이블이 숫자여도 레이블로 찾을 때는 `loc`를 씁니다.

### 3.4 슬라이싱의 끝 처리

In [5]:
print(wait.iloc[1:3])
print(wait.loc[1002:1004])

1002    4.1
1003    5.4
dtype: float64
1002    4.1
1003    5.4
1004    9.2
dtype: float64


**코드 설명**
- `wait.iloc[1:3]`은 위치 1 이상 3 **미만**의 값을 가져옵니다.
- `wait.loc[1002:1004]`는 레이블 1002부터 1004까지를 가져오며 끝 레이블을 **포함**합니다.

**결과 해석**
- `iloc[1:3]`은 2개, `loc[1002:1004]`는 3개입니다. 두 방식의 끝 처리가 다릅니다.

### 3.5 조건으로 값 추출

In [6]:
print(wait[wait >= 9])
print(wait[(wait >= 4) & (wait < 8)])

1004     9.2
1005    10.5
dtype: float64
1002    4.1
1003    5.4
1006    7.3
1007    4.0
1008    4.8
dtype: float64


**코드 설명**
- `wait >= 9`는 각 값이 9 이상인지 참·거짓으로 나타낸 Series를 만듭니다. 이를 대괄호에 넣으면 참인 값만 남습니다.
- 두 번째는 두 조건을 `&`(그리고)로 연결합니다. 각 조건을 **괄호로 묶어야** 하며, `|`는 "또는"입니다.

**결과 해석**
- 9분 이상인 주문은 1004, 1005번 두 건입니다.
- 4분 이상 8분 미만인 주문은 다섯 건입니다.

### 3.6 산술 연산과 통계 메서드

In [7]:
wait_seconds = wait * 60
print(wait_seconds.loc[1001])

print("평균:", wait.mean())
print("중앙값:", wait.median())
print("최댓값:", wait.max(), "→ 주문", wait.idxmax())

192.0
평균: 6.0625
중앙값: 5.1
최댓값: 10.5 → 주문 1005


**코드 설명**
- `wait * 60`은 모든 값에 60을 곱한 **새 Series**를 만들어 `wait_seconds`에 저장합니다. `wait` 자체는 바뀌지 않습니다.
- `.loc[1001]`로 1001번 주문의 환산값을 확인합니다.
- `mean()`, `median()`, `max()`는 값 전체를 하나의 숫자로 요약하고, `idxmax()`는 가장 큰 값의 **레이블**을 돌려줍니다.

**결과 해석**
- 3.2분 × 60 = 192초입니다.
- 평균은 중앙값보다 큽니다(6.0625 > 5.1). 큰 값 하나가 평균을 끌어올리기 때문입니다.
- 가장 오래 기다린 주문은 1005번(10.5분)입니다.

### 3.7 대입과 copy()

`b = a`는 복사가 아니라 같은 객체를 가리킵니다. 실행 전에 `a`와 `c`가 어떻게 될지 예상해 봅니다.

In [8]:
a = pd.Series([3.2, 4.1], index=[1001, 1002])
b = a
c = a.copy()

b.loc[1001] = 99.0
print("a:", a.loc[1001])
print("c:", c.loc[1001])

a: 99.0
c: 3.2


**코드 설명**
- `b = a`는 `a`와 같은 객체에 이름을 하나 더 붙입니다. `c = a.copy()`는 값이 같은 **새 객체**를 만듭니다.
- `b.loc[1001] = 99.0`은 레이블 1001의 값을 99.0으로 바꿉니다.
- `print`로 `a`와 `c`의 같은 위치 값을 비교합니다.

**결과 해석**
- `b`를 바꿨는데 `a`도 99.0으로 바뀌었고, `copy()`로 만든 `c`는 원래 값 3.2를 유지합니다.
- 분석에서는 원본을 그대로 두고, 작업한 결과를 새 이름에 저장합니다.

### 3.8 잠깐 확인하기

- `wait.iloc[1:3]`과 `wait.loc[1002:1004]`는 각각 몇 개의 값을 가져옵니까?
- `wait * 60`을 실행한 뒤 `wait`의 값은 어떻게 됩니까?

## 4. 데이터 미리보기와 구조 확인

각 행이 무엇을 뜻하는지부터 확인합니다. 이 데이터에서 한 행은 **주문 한 건**입니다.

### 4.1 head()

In [9]:
data.head()

,order_id,order_date,weekday,order_hour,menu_name,order_count,wait_minutes,payment_method
0,1001,2026-10-05,월,9,아메리카노,1,3.2,카드
1,1002,2026-10-05,월,10,카페라테,1,4.1,간편결제
2,1003,2026-10-05,월,11,아메리카노,2,5.4,카드
3,1004,2026-10-05,월,12,카페라테,2,9.2,카드
4,1005,2026-10-05,월,12,바닐라라테,1,10.5,간편결제


**코드 설명**
- `data.head()`는 처음 다섯 행을 보여 줍니다.

**결과 해석**
- 열은 주문 번호, 날짜, 요일, 시간대, 메뉴, 수량, 대기시간, 결제 수단입니다.

### 4.2 tail()

In [10]:
data.tail(3)

,order_id,order_date,weekday,order_hour,menu_name,order_count,wait_minutes,payment_method
36,1037,2026-10-09,금,14,아메리카노,1,4.2,카드
37,1038,2026-10-09,금,15,카페라테,2,5.5,간편결제
38,1038,2026-10-09,금,15,카페라테,2,5.5,간편결제


**코드 설명**
- `data.tail(3)`은 마지막 세 행을 보여 줍니다. 괄호 안의 숫자로 행 수를 정합니다.

**결과 해석**
- 1038번 주문이 두 번 나옵니다. 뒤에서 중복값으로 다시 확인합니다.

### 4.3 sample()

In [11]:
data.sample(3, random_state=10)

,order_id,order_date,weekday,order_hour,menu_name,order_count,wait_minutes,payment_method
33,1034,2026-10-09,금,11,카페라테,1,6.0,카드
34,1035,2026-10-09,금,12,아메리카노,3,45.0,카드
2,1003,2026-10-05,월,11,아메리카노,2,5.4,카드


**코드 설명**
- `data.sample(3, random_state=10)`은 무작위로 세 행을 뽑습니다.
- `random_state=10`은 난수의 시작값을 고정해 실행할 때마다 같은 행이 뽑히게 합니다.

**결과 해석**
- 처음과 끝만 보면 중간에 있는 문제를 놓칠 수 있어 무작위 행도 함께 확인합니다.
- 행이 39개라는 것은 주문이 39건이라는 뜻입니다.

### 4.4 데이터 크기와 자료형

In [12]:
print("데이터 크기:", data.shape)
print("열 이름:", data.columns.tolist())
print(data.dtypes)

데이터 크기: (39, 8)
열 이름: ['order_id', 'order_date', 'weekday', 'order_hour', 'menu_name', 'order_count', 'wait_minutes', 'payment_method']
order_id            int64
order_date            str
weekday               str
order_hour          int64
menu_name             str
order_count         int64
wait_minutes      float64
payment_method        str
dtype: object


**코드 설명**
- `data.shape`는 `(행 수, 열 수)`, `data.columns`는 열 이름, `data.dtypes`는 열마다 자료형입니다.
- `columns.tolist()`로 열 이름을 리스트로 바꿔 한 줄에 출력합니다.

**결과 해석**
- `wait_minutes`만 `float64`이고 `order_date`는 날짜처럼 보이지만 문자열입니다. 자료형은 앞으로 할 계산에 맞아야 합니다.

### 4.5 info()

In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        39 non-null     int64  
 1   order_date      39 non-null     str    
 2   weekday         39 non-null     str    
 3   order_hour      39 non-null     int64  
 4   menu_name       39 non-null     str    
 5   order_count     39 non-null     int64  
 6   wait_minutes    38 non-null     float64
 7   payment_method  39 non-null     str    
dtypes: float64(1), int64(3), str(4)
memory usage: 3.8 KB


**코드 설명**
- `data.info()`는 행 수, 열 이름, 비어 있지 않은 값의 개수(`Non-Null Count`), 자료형(`Dtype`)을 한 번에 보여 줍니다.

**결과 해석**
- `wait_minutes`만 `38 non-null`입니다. 전체 39행 중 하나가 비어 있다는 뜻입니다.
- 결측값(`NaN`)을 정수로 표현할 수 없어서 `wait_minutes`가 `float64`로 읽혔습니다.

## 5. 변수 선택과 필터링

### 5.1 열 선택하기

In [14]:
selected = data[["order_hour", "wait_minutes"]]
selected.head()

,order_hour,wait_minutes
0,9,3.2
1,10,4.1
2,11,5.4
3,12,9.2
4,12,10.5


**코드 설명**
- `data[["order_hour", "wait_minutes"]]`에서 안쪽 대괄호는 열 이름의 **목록**, 바깥 대괄호는 선택입니다.
- 결과를 `selected`에 저장하고 `head()`로 앞부분을 확인합니다.

**결과 해석**
- 질문에 필요한 열 두 개만 남은 새 DataFrame이 만들어졌습니다.

### 5.2 열 하나는 Series, unique()

In [15]:
print(type(data["wait_minutes"]))
print(data["menu_name"].unique())

<class 'pandas.Series'>
<ArrowStringArray>
['아메리카노', '카페라테', '바닐라라테', '녹차']
Length: 4, dtype: str


**코드 설명**
- `data["wait_minutes"]`처럼 열 이름 하나만 쓰면 Series가 되며, `type()`으로 확인합니다.
- `unique()`는 중복을 제거한 고유한 값을 보여 줍니다.

**결과 해석**
- 열 하나는 Series이므로 3장의 도구를 그대로 쓸 수 있습니다.
- 메뉴는 네 종류입니다.

### 5.3 행과 열을 함께 선택하기

In [16]:
print(data.loc[0:2, ["order_id", "menu_name"]])
print(data.iloc[0:3, 0:2])

   order_id menu_name
0      1001     아메리카노
1      1002      카페라테
2      1003     아메리카노
   order_id  order_date
0      1001  2026-10-05
1      1002  2026-10-05
2      1003  2026-10-05


**코드 설명**
- `loc[0:2, ["order_id", "menu_name"]]`는 행 레이블 0~2와 열 이름 두 개를 고릅니다. 앞은 행, 뒤는 열입니다.
- `iloc[0:3, 0:2]`는 행 위치 0~2와 열 위치 0, 1을 고릅니다. 끝은 포함하지 않습니다.

**결과 해석**
- 두 코드 모두 행이 3개입니다. `loc[0:2]`는 끝을 포함하고 `iloc[0:3]`은 끝을 제외하기 때문입니다.
- 열은 `loc`가 이름으로(`order_id`, `menu_name`), `iloc`가 위치로(`order_id`, `order_date`) 고릅니다.

### 5.4 조건으로 행 추출하기

In [17]:
lunch_orders = data[data["order_hour"] == 12]
lunch_orders[["order_id", "menu_name", "wait_minutes"]]

,order_id,menu_name,wait_minutes
3,1004,카페라테,9.2
4,1005,바닐라라테,10.5
11,1012,아메리카노,11.4
12,1013,바닐라라테,12.0
19,1020,아메리카노,10.1
20,1021,녹차,NaN
27,1028,바닐라라테,13.2
34,1035,아메리카노,45.0


**코드 설명**
- `data["order_hour"] == 12`는 행마다 12시인지 참·거짓으로 나타냅니다. `data[ 조건 ]`은 참인 행만 가져옵니다.
- 결과를 `lunch_orders`에 저장하고, `[[...]]`로 필요한 열 세 개만 봅니다.

**결과 해석**
- 12시 주문은 8건입니다.

### 5.5 두 조건 함께 쓰기

In [18]:
busy_orders = data[
    (data["order_hour"] >= 12)
    & (data["wait_minutes"] >= 8)
]

busy_orders[["order_hour", "menu_name", "wait_minutes"]]

,order_hour,menu_name,wait_minutes
3,12,카페라테,9.2
4,12,바닐라라테,10.5
11,12,아메리카노,11.4
12,12,바닐라라테,12.0
13,13,카페라테,8.1
19,12,아메리카노,10.1
21,13,카페라테,8.7
27,12,바닐라라테,13.2
34,12,아메리카노,45.0
35,13,바닐라라테,8.4


**코드 설명**
- `data["order_hour"] >= 12`와 `data["wait_minutes"] >= 8`을 각각 괄호로 묶고 `&`로 연결합니다.
- 조건이 길 때는 괄호 안에서 줄을 나누면 읽기 쉽습니다.

**결과 해석**
- 두 조건을 모두 만족하는 행만 남습니다. 결과가 비어 있다면 조건이 너무 좁거나 그런 사례가 없다는 뜻입니다.

## 6. 새로운 열, 정렬, 열 삭제

### 6.1 새로운 열 만들기

기존 열로 새로운 열을 만들 때는 원본을 보존하기 위해 `copy()`한 뒤 작업합니다.

In [19]:
orders = data.copy()
orders["wait_per_item"] = (orders["wait_minutes"] / orders["order_count"]).round(2)

orders[["order_id", "order_count", "wait_minutes", "wait_per_item"]].head()

,order_id,order_count,wait_minutes,wait_per_item
0,1001,1,3.2,3.2
1,1002,1,4.1,4.1
2,1003,2,5.4,2.7
3,1004,2,9.2,4.6
4,1005,1,10.5,10.5


**코드 설명**
- `data.copy()`로 복사본 `orders`를 만듭니다.
- `orders["wait_per_item"] = ...`처럼 없는 열 이름에 값을 대입하면 새 열이 만들어집니다.
- `wait_minutes / order_count`는 같은 행의 값끼리 나누고, `.round(2)`는 소수 둘째 자리까지 반올림합니다.

**결과 해석**
- 1003번은 2잔에 5.4분이므로 잔당 2.7분입니다.

### 6.2 정렬로 이상값 후보 확인

In [20]:
data.sort_values("wait_minutes", ascending=False)[
    ["order_id", "order_hour", "menu_name", "wait_minutes"]
].head()

,order_id,order_hour,menu_name,wait_minutes
34,1035,12,아메리카노,45.0
27,1028,12,바닐라라테,13.2
12,1013,12,바닐라라테,12.0
11,1012,12,아메리카노,11.4
4,1005,12,바닐라라테,10.5


**코드 설명**
- `sort_values("wait_minutes", ascending=False)`는 대기시간이 큰 행부터 정렬합니다.
- `[[...]]`로 필요한 열만 고르고 `.head()`로 위 다섯 행만 봅니다.

**결과 해석**
- 가장 위에 `45.0`분이 있고 그 아래는 13분대입니다. 나머지와 차이가 매우 큽니다.
- 실제로 긴 대기였는지 기록 오류인지는 코드로 알 수 없으므로 **이상값 후보**로 표시합니다.

### 6.3 열 삭제하기

In [21]:
slim = data.drop(columns=["order_date", "weekday"])

print("삭제한 결과:", slim.columns.tolist())
print("원본       :", data.columns.tolist())

삭제한 결과: ['order_id', 'order_hour', 'menu_name', 'order_count', 'wait_minutes', 'payment_method']
원본       : ['order_id', 'order_date', 'weekday', 'order_hour', 'menu_name', 'order_count', 'wait_minutes', 'payment_method']


**코드 설명**
- `data.drop(columns=["order_date", "weekday"])`는 두 열을 뺀 **새 DataFrame**을 돌려줍니다.
- 결과를 `slim`에 저장하고, `slim`과 원본 `data`의 열 이름을 각각 출력해 비교합니다.

**결과 해석**
- `drop()`의 결과를 저장하지 않으면 원본은 그대로입니다. `data`에는 두 열이 남아 있습니다.

### 6.4 잠깐 확인하기

- `data["wait_minutes"]`와 `data[["wait_minutes"]]`의 결과는 어떻게 다릅니까?
- `data.drop(columns=["weekday"])`를 실행했는데 `data`에 `weekday`가 그대로 있는 이유는 무엇입니까?

## 7. 결측값, 중복값, 이상값 처리

### 7.1 결측값 찾기

In [22]:
data.isna().sum()

order_id          0
order_date        0
weekday           0
order_hour        0
menu_name         0
order_count       0
wait_minutes      1
payment_method    0
dtype: int64

**코드 설명**
- `data.isna()`는 값마다 비어 있으면 `True`로 나타낸 표를 만듭니다.
- `.sum()`은 `True`의 개수를 열마다 더합니다.

**결과 해석**
- `wait_minutes`가 1건 비어 있습니다.

### 7.2 중복값 찾기

In [23]:
print("중복 행 수:", data.duplicated().sum())
data[data.duplicated(keep=False)]

중복 행 수: 1


,order_id,order_date,weekday,order_hour,menu_name,order_count,wait_minutes,payment_method
37,1038,2026-10-09,금,15,카페라테,2,5.5,간편결제
38,1038,2026-10-09,금,15,카페라테,2,5.5,간편결제


**코드 설명**
- `data.duplicated()`는 앞에 나온 행과 모든 값이 같은 행을 `True`로 표시하고, `.sum()`으로 개수를 셉니다.
- `keep=False`를 주면 겹친 행을 **모두** 보여 줍니다.

**결과 해석**
- 1038번 주문이 두 번 기록되어 있습니다. 주문 번호까지 같으므로 같은 주문이 두 번 입력되었을 가능성이 큽니다.

### 7.3 처리 방법별 결과 비교

결측이 있는 12시 주문의 평균을 세 가지 방법으로 계산해 비교합니다. 실행하기 전에 어느 값이 가장 낮을지 예상해 봅니다.

In [24]:
h12 = data.drop_duplicates()
h12 = h12[h12["order_hour"] == 12]["wait_minutes"]

print("행 제외        :", round(h12.mean(), 2))
print("0으로 채움     :", round(h12.fillna(0).mean(), 2))
print("중앙값으로 채움:", round(h12.fillna(h12.median()).mean(), 2))

행 제외        : 15.91
0으로 채움     : 13.92
중앙값으로 채움: 15.35


**코드 설명**
- `drop_duplicates()`로 중복 행을 제거하고, 12시 행의 `wait_minutes`만 꺼내 `h12`에 저장합니다.
- `h12.mean()`은 결측을 제외하고 평균을 구합니다. `fillna(0)`은 결측을 0으로, `fillna(h12.median())`은 중앙값으로 채운 뒤 평균을 구합니다.
- `round(..., 2)`는 소수 둘째 자리까지 반올림합니다.

**결과 해석**
- 결측 행을 제외하면 15.91분, 0으로 채우면 13.92분, 중앙값으로 채우면 15.35분입니다.
- 0으로 채우면 존재하지 않은 "대기 0분" 주문이 생겨 평균이 낮아집니다. 이 장에서는 **결측 행을 제외**합니다.

### 7.4 중복값과 결측값 제거하기

정제한 결과는 원본 `data`를 고치지 않고 새 이름 `analysis_data`에 저장합니다.

In [25]:
analysis_data = data.drop_duplicates().dropna(subset=["wait_minutes"]).copy()

print("처리 전 행 수:", len(data))
print("처리 후 행 수:", len(analysis_data))

처리 전 행 수: 39
처리 후 행 수: 37


**코드 설명**
- `drop_duplicates()`로 중복 행을 하나만 남기고, 이어서 `dropna(subset=["wait_minutes"])`로 대기시간이 비어 있는 행을 제외합니다. 점(`.`)으로 이어 한 줄에 씁니다.
- `.copy()`로 독립된 DataFrame을 만들어 `analysis_data`에 저장합니다.
- `len()`은 행의 개수를 돌려줍니다.

**결과 해석**
- 39건에서 중복 1건과 결측 1건이 빠져 37건이 남았습니다.
- 어떤 행을 왜 제외했는지 기록해 두어야 같은 분석을 다른 사람이 반복할 수 있습니다. 이후의 요약과 비교는 모두 `analysis_data`로 계산합니다.

### 7.5 이상값 후보 확인

이 장에서는 계산 없이 쓸 수 있는 30분을 기준으로 정합니다. (IQR 상한은 13.95분, 평균 + 2 × 표준편차는 21.05분이며 두 기준 모두 45분만 걸러냅니다.)

In [26]:
check_needed = analysis_data[
    (analysis_data["wait_minutes"] < 0)
    | (analysis_data["wait_minutes"] > 30)
]

check_needed

,order_id,order_date,weekday,order_hour,menu_name,order_count,wait_minutes,payment_method
34,1035,2026-10-09,금,12,아메리카노,3,45.0,카드


**코드 설명**
- `wait_minutes < 0`이거나(`|`) `wait_minutes > 30`인 행을 조건으로 묶어 `analysis_data`에서 추출합니다. 각 조건은 괄호로 묶습니다.
- 음수는 기록 오류, 30분 초과는 지나치게 긴 대기를 찾는 조건입니다.

**결과 해석**
- 30분을 넘는 주문은 1035번 한 건(45분)입니다.
- 지우지 않고 표시만 해 두었다가 요약에서 영향을 확인합니다.

## 8. 기술통계

### 8.1 수치형 변수 요약

In [27]:
analysis_data[["order_count", "wait_minutes"]].describe()

,order_count,wait_minutes
count,37.000000,37.000000
mean,1.486486,7.178378
std,0.650710,6.935702
min,1.000000,2.900000
25%,1.000000,4.200000
50%,1.000000,5.200000
75%,2.000000,8.100000
max,3.000000,45.000000


**코드 설명**
- `analysis_data[["order_count", "wait_minutes"]]`로 수치형 열 두 개를 고르고, `describe()`로 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 한 번에 구합니다.

**결과 해석**
- `count` 37, `mean` 7.18, `50%`(중앙값) 5.2, `75%` 8.1입니다.
- `max`(45.0)가 `75%`(8.1)보다 훨씬 큽니다. 대부분 8분 이하인데 하나만 45분이라는 뜻이고, 이 값이 평균을 끌어올릴 수 있습니다.

### 8.2 평균과 중앙값 함께 보기

In [28]:
print("평균 대기시간:", round(analysis_data["wait_minutes"].mean(), 2))
print("중앙 대기시간:", analysis_data["wait_minutes"].median())

평균 대기시간: 7.18
중앙 대기시간: 5.2


**코드 설명**
- `mean()`은 값을 모두 더해 개수로 나눈 평균, `median()`은 크기순으로 가운데에 있는 값입니다.
- `round(..., 2)`로 평균을 소수 둘째 자리까지 나타냅니다.

**결과 해석**
- 평균(7.18)이 중앙값(5.2)보다 큽니다. 큰 값 몇 개가 평균을 끌어올리고 있다는 신호입니다.

### 8.3 이상값 후보를 분리하면

In [29]:
usual_data = analysis_data[analysis_data["wait_minutes"] <= 30]

print("30분 이하:", len(usual_data), "건")
print("평균  :", round(analysis_data["wait_minutes"].mean(), 2), "→", round(usual_data["wait_minutes"].mean(), 2))
print("중앙값:", analysis_data["wait_minutes"].median(), "→", usual_data["wait_minutes"].median())

30분 이하: 36 건
평균  : 7.18 → 6.13
중앙값: 5.2 → 5.15


**코드 설명**
- `analysis_data["wait_minutes"] <= 30` 조건으로 30분 이하인 행만 `usual_data`에 저장합니다.
- `print`에 값을 쉼표로 이어 쓰면 공백으로 구분되어 출력됩니다. `→`는 문자열로 넣은 기호입니다.
- 같은 계산을 `analysis_data`와 `usual_data`에 각각 적용해 전후를 비교합니다.

**결과 해석**
- 45분을 빼면 평균은 7.18에서 6.13으로 1.05분 줄었지만 중앙값은 5.2에서 5.15로 0.05분만 줄었습니다.
- 평균이 틀린 것이 아니라, **두 값을 함께 보고 차이가 클 때 이유를 확인**하는 것이 요점입니다.

### 8.4 범주형 변수 빈도

문자로 된 열은 평균을 구할 수 없으므로 값마다 몇 번 나왔는지 셉니다.

In [30]:
analysis_data["menu_name"].value_counts()

menu_name
아메리카노    16
카페라테     10
바닐라라테     6
녹차        5
Name: count, dtype: int64

**코드 설명**
- `value_counts()`는 값의 종류별 개수를 많은 순서로 보여 줍니다.

**결과 해석**
- 아메리카노가 16건으로 가장 많습니다.

### 8.5 범주형 변수 비율

In [31]:
menu_ratio = analysis_data["menu_name"].value_counts(normalize=True) * 100
menu_ratio.round(1)

menu_name
아메리카노    43.2
카페라테     27.0
바닐라라테    16.2
녹차       13.5
Name: proportion, dtype: float64

**코드 설명**
- `value_counts(normalize=True)`는 개수 대신 전체에서 차지하는 비율(0~1)을 돌려줍니다.
- `* 100`으로 퍼센트로 바꾸고, `round(1)`로 소수 첫째 자리까지 나타냅니다.

**결과 해석**
- 아메리카노가 43.2%를 차지합니다.
- 빈도가 높다는 것만으로 선호도가 높다고 말할 수는 없습니다. 며칠간의 주문만 포함하고 품절이나 가격 행사 같은 요인은 기록되어 있지 않기 때문입니다.

## 9. 그룹별 집계와 비교

`groupby()`는 같은 값을 가진 행끼리 묶고 각 묶음에 같은 계산을 적용합니다. 분석 질문에 직접 답하는 단계입니다.

### 9.1 시간대별 집계

In [32]:
hourly_wait = (
    analysis_data
    .groupby("order_hour")["wait_minutes"]
    .agg(["count", "mean", "median"])
    .round(2)
    .sort_values("mean", ascending=False)
)

hourly_wait

,count,mean,median
order_hour,,,
12,7,15.91,11.4
13,5,8.08,8.1
11,5,5.84,5.9
15,5,5.16,5.2
14,5,4.32,4.3
10,5,4.20,4.1
9,5,3.24,3.2


**코드 설명**
- `groupby("order_hour")["wait_minutes"]`는 주문 시간대별로 대기시간을 묶습니다.
- `agg(["count", "mean", "median"])`는 묶음마다 개수, 평균, 중앙값을 한 번에 계산합니다.
- `round(2)`로 반올림하고, `sort_values("mean", ascending=False)`로 평균이 큰 시간대부터 정렬합니다. 전체를 괄호로 묶어 줄마다 한 단계씩 적었습니다.

**결과 해석**
- 12시의 평균이 15.91분으로 가장 깁니다.
- `count` 열이 5~7건입니다. 건수가 적은 그룹의 결과를 지나치게 일반화하지 않습니다.

### 9.2 질문의 답

In [33]:
busiest_hour = hourly_wait["mean"].idxmax()
busiest_mean = hourly_wait.loc[busiest_hour, "mean"]

print(f"평균 대기시간이 가장 긴 시간대는 {busiest_hour}시입니다.")
print(f"해당 시간대의 평균 대기시간은 {busiest_mean:.2f}분입니다.")

평균 대기시간이 가장 긴 시간대는 12시입니다.
해당 시간대의 평균 대기시간은 15.91분입니다.


**코드 설명**
- `hourly_wait["mean"].idxmax()`는 평균이 가장 큰 행의 레이블, 즉 시간대를 돌려줍니다.
- `hourly_wait.loc[시간대, "mean"]`으로 그 시간대의 평균을 꺼냅니다.
- `f"...{busiest_hour}..."`는 중괄호 안의 변수 값을 문장에 넣고, `{busiest_mean:.2f}`는 소수 둘째 자리까지 나타냅니다.

**결과 해석**
- 질문의 답은 12시, 평균 15.91분입니다. 다만 이 결과에는 45분이 포함되어 있습니다.

### 9.3 이상값 포함과 제외 비교

"12시가 가장 길다"는 결론이 45분 한 건에 의존하는지, 같은 계산을 두 자료에서 하고 나란히 놓아 확인합니다.

In [34]:
with_all = analysis_data.groupby("order_hour")["wait_minutes"].mean()
without = usual_data.groupby("order_hour")["wait_minutes"].mean()

compare = pd.DataFrame({"포함": with_all, "제외": without}).round(2)
compare["차이"] = (compare["포함"] - compare["제외"]).round(2)
compare

,포함,제외,차이
order_hour,,,
9,3.24,3.24,0.00
10,4.20,4.20,0.00
11,5.84,5.84,0.00
12,15.91,11.07,4.84
13,8.08,8.08,0.00
14,4.32,4.32,0.00
15,5.16,5.16,0.00


**코드 설명**
- `analysis_data`와 `usual_data`에서 같은 `groupby(...).mean()`을 각각 계산합니다.
- `pd.DataFrame({"포함": ..., "제외": ...})`는 두 Series를 열로 나란히 놓은 표를 만듭니다.
- `compare["차이"] = ...`로 두 열의 차이를 새 열로 추가합니다.

**결과 해석**
- 45분을 빼도 **12시가 가장 길다는 순위는 같습니다.**
- 12시 평균은 15.91분에서 11.07분으로 줄고, 13시와의 격차도 7.83분에서 2.99분으로 작아집니다.
- 결론은 유지되지만 "얼마나 길다"는 말은 45분을 어떻게 다루느냐에 따라 달라지므로, 보고서에는 두 결과를 함께 적습니다.

### 9.4 잠깐 확인하기

- `groupby("order_hour")`에서 그룹을 나누는 기준은 무엇입니까?
- 이상값을 빼도 순위가 같다면 이상값은 신경 쓰지 않아도 됩니까?

## 10. 결과 정리

### 10.1 사실, 해석, 한계

코드가 보여 주는 숫자와 해석을 나누어 씁니다.

| 구분 | 이 노트북에서 |
|:---|:---|
| 확인한 사실 | 12시 평균 대기시간이 11.07분(45분 제외)으로 가장 길었다. 평균은 7.18분, 중앙값은 5.2분이다 |
| 해석 | 점심시간 주문 집중이 영향을 주었을 가능성이 있다 |
| 한계 | 한 카페의 5일 자료이고 시간대마다 5~7건이라 다른 기간에 일반화하기 어렵다. 원인(직원 수, 메뉴별 제조 시간)은 자료에 없다 |